<a href="https://colab.research.google.com/github/surkovaolga2005-png/python-ai-surkoa-olga/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

Цель: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

Данные:
- [Традиции и обычаи стран.csv](https://github.com/surkovaolga2005-png/python-ai-surkoa-olga/blob/main/data/Традиции%20и%20обычаи%20стран.csv) — фестивали, традиции и страны из Wikidata

Что мы делаем:

1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [15]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-surkoa-olga"  # 🔹 ИЗМЕНЕНО: было "python-ai-template"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-surkoa-olga
    !git clone -q https://github.com/surkovaolga2005-png/python-ai-surkoa-olga.git    # 🔹 ИЗМЕНЕНО: был URL другого репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-surkoa-olga


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [17]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

# 🔹 ИЗМЕНЕНО: было чтение двух файлов-примеров, теперь — ваш файл с традициями
df_traditions = pd.read_csv("data/Традиции и обычаи стран.csv")  # 🔹 ИЗМЕНЕНО: имя файла и переменная

print("✅ Загружено строк в df_traditions:", len(df_traditions))  # 🔹 ИЗМЕНЕНО: имя переменной
# 🔹 ДОБАВЛЕНО: дополнительная проверка структуры данных
print(df_traditions.head())
print("Столбцы:", df_traditions.columns.tolist())

✅ Загружено строк в df_traditions: 640
                                     type typeLabelRu  traditionsCount  \
0  http://www.wikidata.org/entity/Q132241   фестиваль              549   
1  http://www.wikidata.org/entity/Q132241   фестиваль              275   
2  http://www.wikidata.org/entity/Q132241   фестиваль              256   
3  http://www.wikidata.org/entity/Q132241   фестиваль              255   
4  http://www.wikidata.org/entity/Q132241   фестиваль              188   

  exampleTradition                              country  countryLabelRu  
0       Тамборрада   http://www.wikidata.org/entity/Q29         Испания  
1          Аутфест   http://www.wikidata.org/entity/Q30             США  
2    Havmanndagene   http://www.wikidata.org/entity/Q20        Норвегия  
3        Айстедвод  http://www.wikidata.org/entity/Q145  Великобритания  
4     Series Mania  http://www.wikidata.org/entity/Q142         Франция  
Столбцы: ['type', 'typeLabelRu', 'traditionsCount', 'exampleTradition', 

🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `Традиции и обычаи стран.csv` есть технические столбцы с URL Викиданных, которые полезны для отладки, но мешают простому анализу:

Исходные столбцы:
- `type`, `country` — содержат URL-ссылки на объекты Wikidata (например, `http://www.wikidata.org/entity/Q29`)
- `typeLabelRu`, `countryLabelRu` — содержат читаемые названия на русском языке
- `traditionsCount` — числовой столбец с количеством традиций
- `exampleTradition` — название конкретной традиции

В этом шаге мы:

1. Удаляем столбцы с URL (`type`, `country`), так как для базового анализа нам достаточно читаемых названий;
2. Переименовываем столбцы с суффиксом `LabelRu`:
   - `typeLabelRu` → `type` (например, "фестиваль")
   - `countryLabelRu` → `country` (например, "Испания")
3. Приводим столбец `traditionsCount` к числовому типу `int`.

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `.fillna(0)` — заменяет пропущенные значения на 0;
- `.astype(int)` — переводит столбец к целочисленному типу.

⚠️ **Важно**: после этого шага у вас останутся только "чистые" столбцы для анализа: `type`, `traditionsCount`, `exampleTradition`, `country`. Если позже понадобится ссылка на оригинальную запись в Викиданных — верните столбцы с URL из исходного файла.

In [18]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для данных о традициях

# Работаем с df_traditions — вашим DataFrame с традициями
if "typeLabelRu" in df_traditions.columns:  # 🔹 ИЗМЕНЕНО: проверка по вашему столбцу вместо "filmLabel"

    # 1) Сначала удаляем столбцы с URL Викиданных, чтобы не было конфликтов имён
    cols_to_drop = [col for col in ["type", "country"] if col in df_traditions.columns]  # 🔹 НОВОЕ: фильтрация существующих столбцов
    if cols_to_drop:
        df_traditions = df_traditions.drop(columns=cols_to_drop)
        print(f"🗑️ Удалены технические столбцы: {cols_to_drop}")

    # 2) Переименовываем столбцы: убираем суффикс LabelRu
    df_traditions = df_traditions.rename(columns={
        "typeLabelRu": "type",        # 🔹 ИЗМЕНЕНО: было filmLabel → film
        "countryLabelRu": "country",  # 🔹 ИЗМЕНЕНО: было countryLabel → country
    })
    print("✅ Столбцы переименованы: typeLabelRu → type, countryLabelRu → country")

    # 3) Приводим traditionsCount к числовому типу
    if "traditionsCount" in df_traditions.columns:  # 🔹 НОВОЕ: проверка существования столбца
        df_traditions["traditionsCount"] = pd.to_numeric(
            df_traditions["traditionsCount"], errors="coerce"
        ).fillna(0).astype(int)
        print("✅ traditionsCount преобразован в int")

    print("✅ df_traditions очищен и готов к анализу")

else:
    print("⏭️ df_traditions уже очищен, пропускаем")

# 🔹 НОВОЕ: быстрая проверка результата
print("\n📋 Итоговые столбцы:", df_traditions.columns.tolist())
print(df_traditions.head(3))

print("\n✅ Данные готовы к анализу")

🗑️ Удалены технические столбцы: ['type', 'country']
✅ Столбцы переименованы: typeLabelRu → type, countryLabelRu → country
✅ traditionsCount преобразован в int
✅ df_traditions очищен и готов к анализу

📋 Итоговые столбцы: ['type', 'traditionsCount', 'exampleTradition', 'country']
        type  traditionsCount exampleTradition   country
0  фестиваль              549       Тамборрада   Испания
1  фестиваль              275          Аутфест       США
2  фестиваль              256    Havmanndagene  Норвегия

✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор вашего DataFrame с традициями:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовому столбцу `traditionsCount` (количество традиций по типу/стране).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать сводку по любому DataFrame.

📋 Что мы ожидаем увидеть после очистки:
- Столбцы: `type`, `traditionsCount`, `exampleTradition`, `country`
- Пример строки: `фестиваль | 549 | Тамборрада | Испания`
- Числовая статистика: мин/макс/среднее количество традиций в `traditionsCount`

In [21]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

    # 🔹 НОВОЕ: если есть числовые столбцы — покажем базовую статистику
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Статистика по числовым столбцам ({', '.join(numeric_cols)}):")
        print(df[numeric_cols].describe())

# 🔍 Шаг 3. Обзор данных о традициях

# 🔹 ИЗМЕНЕНО: был вызов для двух DataFrame, теперь — один ваш df_traditions
show_info(df_traditions, "Традиции и обычаи стран (df_traditions)")  # 🔹 ИЗМЕНЕНО: имя и переменная

# 🔹 НОВОЕ: дополнительная проверка на дубликаты и пропуски
print("\n🔎 Быстрая валидация:")
print("Пропуски в столбцах:\n", df_traditions.isna().sum())
print("Уникальные значения в 'type':", df_traditions['type'].nunique())
print("Уникальные значения в 'country':", df_traditions['country'].nunique())


📊 Традиции и обычаи стран (df_traditions)
Размер: (640, 4)
Столбцы: type, traditionsCount, exampleTradition, country

Первые строки:
        type  traditionsCount exampleTradition         country
0  фестиваль              549       Тамборрада         Испания
1  фестиваль              275          Аутфест             США
2  фестиваль              256    Havmanndagene        Норвегия
3  фестиваль              255        Айстедвод  Великобритания
4  фестиваль              188     Series Mania         Франция

📈 Статистика по числовым столбцам (traditionsCount):
       traditionsCount
count       640.000000
mean         10.501563
std          35.254851
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         549.000000

🔎 Быстрая валидация:
Пропуски в столбцах:
 type                  1
traditionsCount       0
exampleTradition    254
country               5
dtype: int64
Уникальные значения в 'type': 6
Уникальные значения в 'country': 272


✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько уникальных типов традиций и стран есть в данных;
- какие страны встречаются чаще всего (Топ‑5 по числу записей);
- какие типы традиций самые популярные (Топ‑10 по числу строк);
- базовую статистику по количеству традиций (`traditionsCount`);
- диапазон значений: минимальное и максимальное число традиций.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому `df_traditions["countryRu"].value_counts().head()` даёт Топ‑5 стран по числу записей.

⚠️ **Важно**: Эта проверка поможет обнаружить:
- страны с аномально большим числом традиций;
- редкие типы традиций (возможно, ошибки классификации);
- пропущенные значения в ключевых столбцах.

In [24]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# 📊 Датасет: Традиции и обычаи стран
print("\n📊 Датасет: Традиции и обычаи стран (df_traditions)")
print("Уникальных типов традиций:", df_traditions["type"].nunique()) # FIX: Changed 'typeRu' to 'type'
print("Уникальных стран:", df_traditions["country"].nunique()) # FIX: Changed 'countryRu' to 'country'
print("Всего записей:", len(df_traditions))

print("\n🗺️ Топ-5 стран по числу записей:")
print(df_traditions["country"].value_counts().head()) # FIX: Changed 'countryRu' to 'country'

print("\n🎭 Топ-10 типов традиций:")
print(df_traditions["type"].value_counts().head(10)) # FIX: Changed 'typeRu' to 'type'

print("\n📈 Статистика по количеству традиций (traditionsCount):")
print(df_traditions["traditionsCount"].describe())

print("\n🔢 Диапазон значений traditionsCount:")
print(f"Минимум: {df_traditions['traditionsCount'].min()}")
print(f"Максимум: {df_traditions['traditionsCount'].max()}")
print(f"Среднее: {df_traditions['traditionsCount'].mean():.2f}")

# 🔹 Бонус: Проверка на пропуски
print("\n❓ Проверка на пропущенные значения:")
print(df_traditions.isnull().sum())

# 🔹 Бонус: Примеры традиций с максимальным количеством
print("\n⭐ Топ-5 записей с наибольшим traditionsCount:")
display(df_traditions.nlargest(5, "traditionsCount")[["type", "country", "traditionsCount", "exampleTradition"]]) # FIX: Changed 'typeRu' to 'type' and 'countryRu' to 'country'


🔍 Быстрая проверка данных

📊 Датасет: Традиции и обычаи стран (df_traditions)
Уникальных типов традиций: 6
Уникальных стран: 272
Всего записей: 640

🗺️ Топ-5 стран по числу записей:
country
Ирландия     6
Австралия    6
Россия       6
Китай        6
Бразилия     6
Name: count, dtype: int64

🎭 Топ-10 типов традиций:
type
фестиваль         189
традиция          188
церемония         105
фольклор           82
ремесло            64
обряд перехода     11
Name: count, dtype: int64

📈 Статистика по количеству традиций (traditionsCount):
count    640.000000
mean      10.501563
std       35.254851
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max      549.000000
Name: traditionsCount, dtype: float64

🔢 Диапазон значений traditionsCount:
Минимум: 1
Максимум: 549
Среднее: 10.50

❓ Проверка на пропущенные значения:
type                  1
traditionsCount       0
exampleTradition    254
country               5
dtype: int64

⭐ Топ-5 записей с наибольшим traditionsCo

,type,country,traditionsCount,exampleTradition
0,фестиваль,Испания,549,Тамборрада
1,фестиваль,США,275,Аутфест
2,фестиваль,Норвегия,256,Havmanndagene
3,фестиваль,Великобритания,255,Айстедвод
4,фестиваль,Франция,188,Series Mania


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨